In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# subroutines that are shared between both files are here to make it more readable

# Use a quantum circuit to generate a random bit
def random():
    q = QuantumCircuit(1)
    q.h(0)
    q.measure_all()
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)
    return counts.get("1",0)

# ^ from QC-Lab4-A-Complete.ipynb

# Generate 1 2 or 3
def randomChoice():
    q = QuantumCircuit(1)
    q.unitary([[1/math.sqrt(3), -math.sqrt(2/3)],[math.sqrt(2/3), 1/math.sqrt(3)]], [0])
    q.measure_all()

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)

    if counts.get("0",0) == 1:
        return 1
    else:
        if random() == 0:
            return 2
        else:
            return 3

# ^ slightly adjusted from QC-Lab4-A-Complete.ipynb

# Convert to 1 or -1
def pm(x):
    if x == 0:
        return 1
    else:
        return -1

# Prepare singlet state on an existing circuit
def singletCircuit(q):
    q.x(1)
    q.h(0)
    q.cx(0,1)
    q.z(0)

# alice measurement bases 1=X 2=W 3=Z
def measureAlice(q, basis):
    if basis == 1:
        q.h(0)
    elif basis == 2:
        q.ry(-math.pi/4,0)

# bob measurement bases 1=W 2=Z 3=V
def measureBob(q, basis):
    if basis == 1:
        q.ry(-math.pi/4,1)
    elif basis == 3:
        q.ry(math.pi/4,1)

In [5]:
def measureAttacker(q, basis):
    if basis == 1:
        q.ry(-math.pi/4,1)
    elif basis == 3:
        q.ry(math.pi/4,1)

def undoAttacker(q, basis):
    if basis == 1:
        q.ry(math.pi/4,1)
    elif basis == 3:
        q.ry(-math.pi/4,1)

def measurePair(aBasis,bBasis,eBasis):
    q = QuantumCircuit(2,3)

    singletCircuit(q)

    measureAttacker(q,eBasis)
    q.measure(1,0)
    undoAttacker(q,eBasis)

    measureAlice(q,aBasis)
    measureBob(q,bBasis)

    q.measure(0,1)
    q.measure(1,2)

    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result()
    counts = result_sim.get_counts(compiled)

    result = list(counts.keys())[0]

    bobBit = int(result[0])
    aliceBit = int(result[1])

    return [aliceBit,bobBit]

N = 100

aliceBases = []
bobBases = []
attackerBases = []

aliceBits = []
bobBits = []

aliceKey = []
bobKey = []

totalXW = 0
countXW = 0

totalXV = 0
countXV = 0

totalZW = 0
countZW = 0

totalZV = 0
countZV = 0

while len(aliceKey) < N:
    a = randomChoice()
    b = randomChoice()
    e = randomChoice()

    bits = measurePair(a,b,e)

    x = bits[0]
    y = bits[1]

    aliceBases = aliceBases + [a]
    bobBases = bobBases + [b]
    attackerBases = attackerBases + [e]

    aliceBits = aliceBits + [x]
    bobBits = bobBits + [y]

    if a == 2 and b == 1:
        aliceKey = aliceKey + [x]
        bobKey = bobKey + [1-y]

    elif a == 3 and b == 2:
        aliceKey = aliceKey + [x]
        bobKey = bobKey + [1-y]

    elif a == 1 and b == 1:
        totalXW = totalXW + pm(x)*pm(y)
        countXW = countXW + 1

    elif a == 1 and b == 3:
        totalXV = totalXV + pm(x)*pm(y)
        countXV = countXV + 1

    elif a == 3 and b == 1:
        totalZW = totalZW + pm(x)*pm(y)
        countZW = countZW + 1

    elif a == 3 and b == 3:
        totalZV = totalZV + pm(x)*pm(y)
        countZV = countZV + 1

print("Alice bases: ",aliceBases)
print("Bob bases: ",bobBases)
print("Attacker bases: ",attackerBases)

print("Alice bits: ",aliceBits)
print("Bob bits: ",bobBits)

print("Alice key: ",aliceKey)
print("Bob key: ",bobKey)

EXW = totalXW/countXW
EXV = totalXV/countXV
EZW = totalZW/countZW
EZV = totalZV/countZV

S = abs(EXW - EXV + EZW + EZV)

print("<X ⊗ W> =",EXW)
print("<X ⊗ V> =",EXV)
print("<Z ⊗ W> =",EZW)
print("<Z ⊗ V> =",EZV)

print("S =",S)
print("2 =",2)
print("2√2 =",2*math.sqrt(2))

if S < 2:
    print("Attack detected")
else:
    print("No attack detected")

print("Counts:")
print("XW:",countXW)
print("XV:",countXV)
print("ZW:",countZW)
print("ZV:",countZV)
print("Total rounds:",len(aliceBases))

Alice bases:  [2, 2, 1, 1, 2, 1, 2, 3, 3, 3, 2, 3, 2, 2, 2, 1, 2, 2, 2, 2, 2, 3, 3, 2, 2, 3, 3, 2, 2, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 3, 3, 2, 2, 1, 1, 3, 2, 1, 1, 1, 1, 2, 3, 3, 1, 3, 2, 3, 1, 2, 3, 3, 2, 3, 3, 3, 2, 3, 3, 3, 3, 1, 1, 3, 2, 3, 3, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 3, 3, 3, 3, 2, 1, 1, 2, 1, 2, 1, 1, 1, 3, 2, 3, 3, 2, 3, 3, 1, 2, 2, 3, 1, 3, 1, 2, 3, 3, 2, 3, 1, 3, 1, 2, 3, 2, 3, 1, 1, 3, 2, 3, 1, 1, 1, 1, 3, 2, 1, 1, 2, 3, 3, 3, 1, 1, 1, 3, 3, 2, 3, 2, 3, 1, 2, 2, 3, 2, 2, 2, 2, 2, 3, 2, 1, 3, 3, 1, 2, 2, 1, 3, 1, 2, 3, 3, 2, 3, 3, 1, 2, 3, 3, 3, 2, 3, 3, 2, 2, 3, 1, 1, 3, 1, 3, 1, 3, 2, 3, 1, 1, 1, 2, 2, 2, 1, 3, 2, 1, 2, 1, 2, 2, 3, 3, 2, 1, 2, 2, 3, 3, 2, 2, 2, 3, 2, 3, 2, 3, 1, 1, 3, 1, 2, 3, 3, 3, 2, 3, 2, 3, 2, 3, 1, 2, 2, 3, 2, 2, 1, 2, 1, 1, 3, 1, 2, 1, 2, 3, 1, 1, 1, 3, 1, 2, 2, 3, 3, 3, 2, 1, 3, 1, 2, 3, 2, 2, 3, 1, 2, 2, 1, 2, 1, 3, 1, 1, 3, 3, 3, 3, 2, 1, 1, 3, 3, 1, 2, 2, 3, 1, 2, 3, 3, 1, 3, 2, 3, 3, 2, 3, 3, 3, 1, 2, 1, 3, 2, 2, 3, 3, 2, 3, 1, 1, 1, 2, 2, 2, 3

For cases where Alice and Bob measured in the same basis the results were opposite as expected.

Bell parameter was calculated as approximately S = 1.53

This is below the classical limit of S <= 2, meaning the expected quantum correlations are lost.

This indicates that the entanglement has been disturbed.

Attack detected as expected when an attacker interferes with the qubits.